## Lecture 6: Distributed Computing 1

****Pre-check:** `python -c "import dask; print(dask.__version__)"` — should print a version number**

In [1]:
!python -c "import dask; print(dask.__version__)"

2026.1.2


### ****Exercise 1:** Dask delayed — lazy evaluation & Monte Carlo π**

**Warm up with `dask.delayed`: understand what lazy evaluation means in practice.**

In [ ]:
# From slide 28

import dask
import random
import time
import statistics
from dask import delayed


def monte_carlo_chunk(n_samples):
    inside = 0
    for _ in range(n_samples):
        x, y = random.random(), random.random()
        if x*x + y*y <= 1:
            inside += 1
    return inside


total, n_chunks = 1_000_000, 8
samples = total // n_chunks

# Serial baseline
t0 = time.perf_counter()
results = [monte_carlo_chunk(samples) for _ in range(n_chunks)]
t_serial = time.perf_counter() - t0
print(f"Serial: {t_serial:.3f} s pi={4*sum(results)/total:.4f}")

# Dask delayed -- task graph is built, not executed yet
tasks = [delayed(monte_carlo_chunk)(samples) for _ in range(n_chunks)]
t0 = time.perf_counter()
results = dask.compute(*tasks)
t_dask = time.perf_counter() - t0
print(f"Dask:   {t_dask:.3f} s pi={4*sum(results)/total:.4f}")

# Visualise (requires: conda install python-graphviz)
dask.visualize(*tasks, filename='task_graph.png')

****Note:** without a `LocalCluster`, `dask.compute()` uses a single-threaded scheduler — Dask may be *slower* than serial here; that is expected. The point of E1 is lazy evaluation and task graph structure, not speedup. E2 adds a LocalCluster.**

**1. Run the serial Monte Carlo π (see code example) and note the time.**

**2. Wrap monte carlo chunk with `delayed`; call it `n_chunks` times. Observe: the function does not execute yet — no output.**

**3. Visualise the task graph: `dask.visualize(*tasks, filename=’graph.png’)` Open the PNG — which tasks can run in parallel?**

**4. Call `dask.compute(*tasks)` and time it. Compare to serial.**

**5. Vary `n_chunks` (4, 8, 16, 32): how does overhead scale with very small chunks?**

****Done?** What speedup (or slowdown) did you observe, and why? → discuss with a neighbour**

### ****Exercise 2:** LocalCluster & dashboard**

**Use a real Dask scheduler and watch task execution live.**

In [ ]:
# From slide 30

from dask.distributed import Client, LocalCluster
import dask
from dask import delayed

# Create local cluster; start with max workers -- scale() adjusts without restarting
cluster = LocalCluster(n_workers=8, threads_per_worker=1)
client = Client(cluster)

print(f"Dashboard: {client.dashboard_link}")
# --> open the printed URL in your browser

# Rerun E1 tasks; LocalCluster scheduler takes over
tasks = [delayed(monte_carlo_chunk)(samples) for _ in range(n_chunks)]
results = dask.compute(*tasks)

# Vary n_workers: scale() resizes without restarting the scheduler
# (recreating LocalCluster while the browser is open breaks the dashboard)
cluster.scale(4)
client.wait_for_workers(4)    # now 4 workers
tasks = [delayed(monte_carlo_chunk)(samples) for _ in range(n_chunks)]
results = dask.compute(*tasks)

client.close()
cluster.close()

****Dashboard panels to explore:****
- ***Task Stream:* real-time execution per worker**
- ***Workers:* CPU and memory per process**
- ***Progress:* completion percentage of current compute**

**1. Create a `LocalCluster` and connect a `Client` (see code example).**

**2. Open the dashboard URL in your browser (printed on startup).**

**3. Rerun E1 Monte Carlo π while watching the **Task Stream** panel**

**4. Try varying `n_workers` (2, 4, 8) — does the task stream change?**

**5. Observe: are all workers busy? Are there any stragglers?**

****Done?** What did you see in the task stream? → discuss with a neighbour**

### ****Exercise 3 (optional):** Dependent task graphs — two-pass normalisation**

**So far every task has been independent. Real pipelines may have **cross-chunk dependencies** — Dask resolves them automatically.**

****Two-pass normalisation** (classic fan-in / fan-out pattern):**
1. ****Stage 1 (parallel):** generate each chunk; compute per-chunk maximum**
2. ****Stage 2 (fan-in):** global max — waits for all Stage 1 tasks**
3. ****Stage 3 (fan-out):** normalise each chunk using global max — parallel again**

**You write zero synchronisation code. Dask infers the execution order from which delayed objects appear as arguments.**

In [ ]:
# From slide 32

import time
import numpy as np
import dask
from dask import delayed
from dask.distributed import Client, LocalCluster

# Each function sleeps so stages are visible in the dashboard Task Stream


@delayed
def generate(seed, n):
    time.sleep(0.3)
    return np.random.default_rng(seed).standard_normal(n)


@delayed
def chunk_max(data):
    time.sleep(0.2)
    return float(np.max(np.abs(data)))


@delayed
def global_max(maxima):
    time.sleep(0.2)
    return max(maxima)  # fan-in: waits for ALL chunk_max


@delayed
def normalise(data, g):
    time.sleep(0.3)
    return data / g     # fan-out: parallel once gmax ready


if __name__ == '__main__':
    cluster = LocalCluster(n_workers=4, threads_per_worker=1)
    client = Client(cluster)
    print(client.dashboard_link)

    chunks = [generate(i, 50_000) for i in range(8)]    # Stage 1a
    maxima = [chunk_max(c) for c in chunks]             # Stage 1b
    gmax = global_max(maxima)                           # Stage 2: fan-in
    normed = [normalise(c, gmax) for c in chunks]       # Stage 3: fan-out

    dask.visualize(*normed, filename='task_graph_pipeline.png')
    t0 = time.perf_counter()
    results = dask.compute(*normed)
    print(f"Wall time: {time.perf_counter()-t0:.2f} s")
    client.close()
    cluster.close()

**What to observe:**

- ****Task graph** (`task_graph_pipeline.png`): does the hourglass shape match the three-stage structure?**

- ****Task Stream — phases:** identify the two parallel bands and the single-task bottleneck match each colour to a function (`generate`, `chunk_max`, `global_max`, `normalise`)**

- ****Task Stream — red boxes (`transfer-*`):** unavoidable — data *must* cross worker boundaries at fan-in/fan-out points. This is the $\beta$*s* term. NumPy minimises $s$; the transfers remain.**

- ****Timing:** how close to $T_{\infty} = T_{\text{S1}}/n_w + T_{\text{S2}} + T_{\text{S3}} /n_w$ ?**

****Done?** Identify the phases in the Task Stream → discuss with a neighbour**

### ****Milestone 1:** Dask Mandelbrot (local)**

****Approach:** wrap your existing `Numba_mandelbrot_chunk` with `dask.delayed` — same row-chunk structure as L04/L05, one task per row-band:**

`dask.compute(*[delayed(f)(args) for args in chunk args])` ← replaces pool.map(f, chunk args)

**Common pitfall:** Do NOT iterate Dask arrays inside the `max_iter` loop (e.g. `Z = da.where(...)`). Each of the 100 iterations becomes a separate task — $∼100,000$ tasks for $N = 1024$ — and overhead dominates. `dask.delayed` wraps the entire Numba function as *one* atomic task per chunk.**

In [ ]:
# From slide 34

from dask import delayed
from dask.distributed import Client, LocalCluster
import dask, numpy as np, time, statistics
# mandelbrot_chunk: your @njit(cache=True) function from L04/L05
def mandelbrot_dask(N, x_min, x_max, y_min, y_max,
                    max_iter=100, n_chunks=32):
    chunk_size = max(1, N // n_chunks)
    tasks, row = [], 0
    while row < N:
        row_end = min(row + chunk_size, N)
        tasks.append(delayed(mandelbrot_chunk)(
            row, row_end, N, x_min, x_max, y_min, y_max, max_iter))
        row = row_end
    parts = dask.compute(*tasks)
    return np.vstack(parts)

if __name__ == '__main__':
    N, max_iter = 1024, 100
    X_MIN, X_MAX, Y_MIN, Y_MAX = -2.5, 1.0, -1.25, 1.25
    cluster = LocalCluster(n_workers=8, threads_per_worker=1)
    client = Client(cluster)
    client.run(lambda: mandelbrot_chunk(0, 8, 8, X_MIN, X_MAX, # warm up all workers
                                        Y_MIN, Y_MAX, 10))
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        result = mandelbrot_dask(N, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter)
        times.append(time.perf_counter() - t0)
    print(f"Dask local (n_chunks=32): {statistics.median(times):.3f} s")
    client.close(); cluster.close()

**1. Implement `mandelbrot_dask()` using `dask.delayed` (see code example).**

**2. Verify: `np.array_equal(ref, result)` against your serial output.**

**3. Warm up Numba JIT in *all* workers with `client.run()` before timing.**

**4. Time 3 runs; record `statistics.median()`.**

****Done?** Record time in performance notebook (MP2) → commit**

### ****Milestone 2:** Chunk size sweep**

**Find the optimal chunk count for Dask local on your machine.**

****Sweep** `n_chunks` over a range guided by the three-way trade-off (see theory slides). For each setting: time 3 runs (median), compute $LIF = p \cdot T_p / T_1 − 1$.**

**1. Adapt your L05 Dask implementation to loop over `n_chunks` values: keep `LocalCluster` open across all measurements (start-up cost paid once).**

**2. Warm up Numba JIT in *all* workers with `client.run()` before the sweep loop.**

**3. Record: wall time and LIF per `n_chunks; plot wall time vs. n_chunks (log scale).**

**4. Note: Dask has higher $\alpha$ per task than multiprocessing — expect the optimal `n_chunks` to be smaller than in L05.**

****Print table columns:****

**`n_chunks | time (s) | vs 1x | speedup | LIF`**

****Record:****

**`n_chunks optimal, t min, LIF min`**

**Save plot as `dask_chunk_sweep.png`**

****Optional:** Add block-level early exit to your Numba kernel: after each iteration step, check if all pixels in the chunk have diverged — if so, break the loop. Chunks entirely outside the Mandelbrot set then skip remaining iterations. Compare timing with and without this optimisation.**

****Done?** Record `n_chunks` optimal and `LIF_min` in performance notebook (MP2) → commit**

### ****Milestone 3:** Full benchmark (all methods)**

**Collect all timings and compare every implementation built so far.**

**Add a Dask row to your performance notebook:**

| **Implementation**      | **Time (s)**        | **Speedup vs. naive** |
| ----------------------- | ------------------- | --------------------- |
| Naive Python            |                     | 1×                    |
| NumPy                   |                     |                       |
| Numba (@njit)           |                     |                       |
| Numba + multiprocessing |                     |                       |
| Dask local              |                     |                       |
| Dask cluster            | (to be added in L7) |                       |


**Reflection questions (address in your written reflection):**

- **How does Dask local compare to multiprocessing at the same worker count?**

- **What does the overhead difference tell you about when to choose each tool?**

****Submit:** Add your Dask local result to the Performance Tracker in Moodle.**

****Done?** Record all results in performance notebook (MP2) → commit**